# Feature Engineering - Ingresos por Período

## Objetivo

Este notebook crea features adicionales para el modelo de predicción de ingresos por período. Las features incluyen:

- Features temporales (día de semana, mes, trimestre)
- Features agregadas (promedios de pedidos e ingresos)
- Features de tendencia y estacionalidad
- Features de festivos
- Normalización y escalado
- División train/validation/test

## Estructura

1. Carga de datos limpios
2. Features temporales
3. Features agregadas
4. Features de tendencia y estacionalidad
5. Features de festivos
6. Normalización y escalado
7. División temporal
8. Validación de calidad

In [7]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from datetime import datetime, timedelta
import holidays
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Configurar rutas
DATA_DIR = Path('../data')

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## 1. Carga de Datos Limpios

Cargamos el dataset limpio de ingresos creado en el notebook de EDA.

In [8]:
# Intentar cargar dataset limpio, si no existe, recrearlo
try:
    df = pd.read_csv(DATA_DIR / 'datos_ingresos_periodo_limpio.csv')
    df['fecha'] = pd.to_datetime(df['fecha'])
    print("✅ Dataset limpio cargado desde archivo")
except FileNotFoundError:
    print("⚠️ Dataset limpio no encontrado, recreando desde datos originales...")
    # Cargar datos originales y procesar
    orders_df = pd.read_csv(DATA_DIR / 'triplekb-orders-2026-03-08T15-37-31.csv')
    orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])
    
    ve_holidays = holidays.Venezuela(years=[2025, 2026])
    orders_df['es_festivo'] = orders_df['order_date'].apply(
        lambda x: 'Sí' if x in ve_holidays else 'No'
    )
    
    ingresos_diarios = orders_df.groupby('order_date').agg({
        'total': 'sum',
        'order_id': 'count',
        'hour_of_day': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.mean()
    }).reset_index()
    
    ingresos_diarios.columns = ['fecha', 'ingresos_totales', 'cantidad_pedidos', 'hora_pico_demanda']
    ingresos_diarios['dia_semana'] = ingresos_diarios['fecha'].dt.day_name()
    ingresos_diarios['mes'] = ingresos_diarios['fecha'].dt.month
    ingresos_diarios['es_fin_semana'] = ingresos_diarios['dia_semana'].apply(
        lambda x: 'Sí' if x in ['Saturday', 'Sunday'] else 'No'
    )
    ingresos_diarios['es_festivo'] = ingresos_diarios['fecha'].apply(
        lambda x: 'Sí' if x in ve_holidays else 'No'
    )
    ingresos_diarios['promedio_pedido'] = ingresos_diarios['ingresos_totales'] / ingresos_diarios['cantidad_pedidos']
    ingresos_diarios['promociones_activas'] = 0
    ingresos_diarios['eventos_especiales'] = 'Normal'
    
    df = ingresos_diarios
    print("✅ Dataset recreado")

print(f"\n📊 Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"📅 Rango de fechas: {df['fecha'].min().strftime('%Y-%m-%d')} a {df['fecha'].max().strftime('%Y-%m-%d')}")

✅ Dataset limpio cargado desde archivo

📊 Dimensiones: 258 filas × 11 columnas
📅 Rango de fechas: 2025-06-23 a 2026-03-08


## 2. Features Temporales

Creamos features temporales para capturar patrones temporales.

In [9]:
# Features temporales básicas
df['año'] = df['fecha'].dt.year
df['trimestre'] = df['fecha'].dt.quarter
df['dia_del_mes'] = df['fecha'].dt.day
df['semana_del_año'] = df['fecha'].dt.isocalendar().week

# Días desde inicio del período
df['dias_desde_inicio'] = (df['fecha'] - df['fecha'].min()).dt.days

# Features cíclicas para día de semana
df['dia_semana_num'] = df['dia_semana'].map({
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
    'Friday': 4, 'Saturday': 5, 'Sunday': 6
})
df['dia_semana_sin'] = np.sin(2 * np.pi * df['dia_semana_num'] / 7)
df['dia_semana_cos'] = np.cos(2 * np.pi * df['dia_semana_num'] / 7)

# Features cíclicas para mes
df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)

print("✅ Features temporales creadas")

✅ Features temporales creadas


## 3. Features Agregadas

Creamos features que capturan promedios y ratios históricos.

In [10]:
# Ordenar por fecha
df = df.sort_values('fecha').reset_index(drop=True)

# Promedio de pedidos últimos N días
for window in [7, 14, 30]:
    df[f'pedidos_promedio_{window}d'] = df['cantidad_pedidos'].shift(1).rolling(window=window, min_periods=1).mean()
    df[f'ingresos_promedio_{window}d'] = df['ingresos_totales'].shift(1).rolling(window=window, min_periods=1).mean()

# Ratio fin de semana (promedio fin de semana / promedio días laborables)
df['ratio_fin_semana'] = df['es_fin_semana'].apply(lambda x: 1.0 if x == 'Sí' else 0.0)

print("✅ Features agregadas creadas")

✅ Features agregadas creadas


## 4. Features de Tendencia y Estacionalidad

Creamos features que capturan tendencias y patrones estacionales.

In [11]:
# Lag features: ingresos de días anteriores
for lag in [1, 7, 14]:
    df[f'lag_ingresos_{lag}d'] = df['ingresos_totales'].shift(lag)
    df[f'lag_pedidos_{lag}d'] = df['cantidad_pedidos'].shift(lag)

# Tendencias: diferencia entre períodos recientes y pasados
df['tendencia_ingresos_7d'] = (
    df['ingresos_totales'].shift(1).rolling(window=7, min_periods=1).mean() - 
    df['ingresos_totales'].shift(8).rolling(window=7, min_periods=1).mean()
)

# Volatilidad
df['volatilidad_ingresos_7d'] = df['ingresos_totales'].shift(1).rolling(window=7, min_periods=1).std()

print("✅ Features de tendencia y estacionalidad creadas")

✅ Features de tendencia y estacionalidad creadas


## 5. Features de Festivos

Creamos features relacionadas con festivos.

In [12]:
# Codificar festivos como binario
df['es_festivo_num'] = df['es_festivo'].map({'Sí': 1, 'No': 0})
df['es_fin_semana_num'] = df['es_fin_semana'].map({'Sí': 1, 'No': 0})

# Días hasta el próximo festivo
ve_holidays = holidays.Venezuela(years=[2025, 2026])
# Convertir Timestamp a date para comparación
fecha_min = df['fecha'].min().date()
fechas_festivos = sorted([fecha for fecha in ve_holidays.keys() if fecha >= fecha_min])

def dias_hasta_proximo_festivo(fecha):
    # Convertir Timestamp a date si es necesario
    fecha_date = fecha.date() if isinstance(fecha, pd.Timestamp) else fecha
    for festivo in fechas_festivos:
        if festivo >= fecha_date:
            return (festivo - fecha_date).days
    return np.nan

df['dias_hasta_proximo_festivo'] = df['fecha'].apply(dias_hasta_proximo_festivo)

# Días desde el último festivo
def dias_desde_ultimo_festivo(fecha):
    # Convertir Timestamp a date si es necesario
    fecha_date = fecha.date() if isinstance(fecha, pd.Timestamp) else fecha
    for festivo in reversed(fechas_festivos):
        if festivo <= fecha_date:
            return (fecha_date - festivo).days
    return np.nan

df['dias_desde_ultimo_festivo'] = df['fecha'].apply(dias_desde_ultimo_festivo)

print("✅ Features de festivos creadas")

✅ Features de festivos creadas


## 6. Normalización y Escalado

Preparamos las features para normalización.

In [13]:
# Seleccionar features numéricas para normalizar
features_numericas = [
    'cantidad_pedidos', 'promedio_pedido', 'hora_pico_demanda',
    'pedidos_promedio_7d', 'pedidos_promedio_14d', 'pedidos_promedio_30d',
    'ingresos_promedio_7d', 'ingresos_promedio_14d', 'ingresos_promedio_30d',
    'lag_ingresos_1d', 'lag_ingresos_7d', 'lag_ingresos_14d',
    'lag_pedidos_1d', 'lag_pedidos_7d', 'lag_pedidos_14d',
    'tendencia_ingresos_7d', 'volatilidad_ingresos_7d',
    'dias_desde_inicio', 'dias_hasta_proximo_festivo', 'dias_desde_ultimo_festivo'
]

# Rellenar valores NaN
for feature in features_numericas:
    if feature in df.columns:
        df[feature] = df[feature].fillna(df[feature].median() if df[feature].notna().sum() > 0 else 0)

print("✅ Features numéricas preparadas para normalización")

✅ Features numéricas preparadas para normalización


## 7. División Temporal Train/Validation/Test

Dividimos los datos temporalmente.

In [14]:
# División temporal
train_end = pd.Timestamp('2025-12-31')
val_end = pd.Timestamp('2026-02-28')

train_df = df[df['fecha'] <= train_end].copy()
val_df = df[(df['fecha'] > train_end) & (df['fecha'] <= val_end)].copy()
test_df = df[df['fecha'] > val_end].copy()

print("=" * 80)
print("DIVISIÓN TEMPORAL")
print("=" * 80)
print(f"\n📊 Train: {len(train_df):,} días ({train_df['fecha'].min().strftime('%Y-%m-%d')} a {train_df['fecha'].max().strftime('%Y-%m-%d')})")
print(f"📊 Validation: {len(val_df):,} días ({val_df['fecha'].min().strftime('%Y-%m-%d')} a {val_df['fecha'].max().strftime('%Y-%m-%d')})")
print(f"📊 Test: {len(test_df):,} días ({test_df['fecha'].min().strftime('%Y-%m-%d')} a {test_df['fecha'].max().strftime('%Y-%m-%d')})")

DIVISIÓN TEMPORAL

📊 Train: 191 días (2025-06-23 a 2025-12-31)
📊 Validation: 59 días (2026-01-01 a 2026-02-28)
📊 Test: 8 días (2026-03-01 a 2026-03-08)


## 8. Normalización de Features

Aplicamos normalización después de la división.

In [15]:
# Seleccionar features numéricas disponibles
features_numericas_disponibles = [f for f in features_numericas if f in df.columns]

# Aplicar normalización
scaler = StandardScaler()

train_df_scaled = train_df.copy()
val_df_scaled = val_df.copy()
test_df_scaled = test_df.copy()

train_df_scaled[features_numericas_disponibles] = scaler.fit_transform(
    train_df[features_numericas_disponibles]
)
val_df_scaled[features_numericas_disponibles] = scaler.transform(
    val_df[features_numericas_disponibles]
)
test_df_scaled[features_numericas_disponibles] = scaler.transform(
    test_df[features_numericas_disponibles]
)

print("✅ Features normalizadas")

✅ Features normalizadas


## 9. Selección de Features Finales

Seleccionamos las features finales para el modelo.

In [16]:
# Seleccionar features finales
feature_columns = [
    # Temporales
    'mes', 'trimestre', 'semana_del_año', 'dias_desde_inicio',
    'dia_semana_sin', 'dia_semana_cos', 'mes_sin', 'mes_cos',
    # Agregadas
    'cantidad_pedidos', 'promedio_pedido', 'hora_pico_demanda',
    'pedidos_promedio_7d', 'pedidos_promedio_14d', 'pedidos_promedio_30d',
    'ingresos_promedio_7d', 'ingresos_promedio_14d', 'ingresos_promedio_30d',
    'ratio_fin_semana',
    # Tendencias
    'lag_ingresos_1d', 'lag_ingresos_7d', 'lag_ingresos_14d',
    'lag_pedidos_1d', 'lag_pedidos_7d', 'lag_pedidos_14d',
    'tendencia_ingresos_7d', 'volatilidad_ingresos_7d',
    # Festivos
    'es_festivo_num', 'es_fin_semana_num',
    'dias_hasta_proximo_festivo', 'dias_desde_ultimo_festivo',
    # Otros
    'promociones_activas'
]

# Filtrar features que existen
feature_columns = [f for f in feature_columns if f in train_df_scaled.columns]

# Crear datasets finales
X_train = train_df_scaled[feature_columns].copy()
y_train = train_df_scaled['ingresos_totales'].copy()

X_val = val_df_scaled[feature_columns].copy()
y_val = val_df_scaled['ingresos_totales'].copy()

X_test = test_df_scaled[feature_columns].copy()
y_test = test_df_scaled['ingresos_totales'].copy()

print("=" * 80)
print("DATASETS FINALES PARA MODELADO")
print("=" * 80)
print(f"\n📊 Features seleccionadas: {len(feature_columns)}")
print(f"\n📊 Dimensiones:")
print(f"   Train: X={X_train.shape}, y={y_train.shape}")
print(f"   Validation: X={X_val.shape}, y={y_val.shape}")
print(f"   Test: X={X_test.shape}, y={y_test.shape}")

# Guardar datasets
X_train.to_csv(DATA_DIR / 'X_train_ingresos.csv', index=False)
y_train.to_csv(DATA_DIR / 'y_train_ingresos.csv', index=False)
X_val.to_csv(DATA_DIR / 'X_val_ingresos.csv', index=False)
y_val.to_csv(DATA_DIR / 'y_val_ingresos.csv', index=False)
X_test.to_csv(DATA_DIR / 'X_test_ingresos.csv', index=False)
y_test.to_csv(DATA_DIR / 'y_test_ingresos.csv', index=False)

print(f"\n✅ Datasets guardados en {DATA_DIR}")

DATASETS FINALES PARA MODELADO

📊 Features seleccionadas: 31

📊 Dimensiones:
   Train: X=(191, 31), y=(191,)
   Validation: X=(59, 31), y=(59,)
   Test: X=(8, 31), y=(8,)

✅ Datasets guardados en ../data


## 10. Validación de Calidad

Verificamos la calidad de los datos preparados.

In [17]:
print("=" * 80)
print("VALIDACIÓN DE CALIDAD DE DATOS")
print("=" * 80)

# Verificar valores faltantes
print(f"\n📊 Valores faltantes:")
print(f"   Train: {X_train.isnull().sum().sum()}")
print(f"   Validation: {X_val.isnull().sum().sum()}")
print(f"   Test: {X_test.isnull().sum().sum()}")

# Estadísticas de la variable objetivo
print(f"\n📊 Estadísticas de ingresos_totales (y):")
print(f"   Train - Media: ${y_train.mean():.2f}, Mediana: ${y_train.median():.2f}")
print(f"   Validation - Media: ${y_val.mean():.2f}, Mediana: ${y_val.median():.2f}")
print(f"   Test - Media: ${y_test.mean():.2f}, Mediana: ${y_test.median():.2f}")

print(f"\n✅ Validación completada")

VALIDACIÓN DE CALIDAD DE DATOS

📊 Valores faltantes:
   Train: 0
   Validation: 0
   Test: 0

📊 Estadísticas de ingresos_totales (y):
   Train - Media: $1131.01, Mediana: $1064.35
   Validation - Media: $1752.01, Mediana: $1536.55
   Test - Media: $1790.59, Mediana: $1612.74

✅ Validación completada


## Resumen

Este notebook ha preparado los datos para el modelado de ingresos por período:

1. ✅ Features temporales creadas
2. ✅ Features agregadas creadas
3. ✅ Features de tendencia y estacionalidad creadas
4. ✅ Features de festivos creadas
5. ✅ Features normalizadas
6. ✅ División temporal realizada
7. ✅ Datasets guardados para modelado

### Próximos Pasos

El siguiente notebook (`06_modelo_ingresos_periodo.ipynb`) utilizará estos datasets para entrenar modelos de predicción de ingresos.